## **SNAP Jupyter demo notebook**
**ESA BIOMASS polarimetric processing — calibration, speckle filtering, decomposition, terrain correction**

In summary, this workflow contains:

- Background on BIOMASS as a fully polarimetric P-band SAR mission and the SNAP polarimetric processing chain
- End-to-end graph: `Read → Calibration → Polarimetric-Matrices → Polarimetric-Speckle-Filter → Polarimetric-Decomposition → Terrain-Correction`
- Three featured decompositions: **H-A-Alpha** (entropy/anisotropy/alpha), **Pauli** (RGB), and **Freeman-Durden** (surface/double-bounce/volume)
- Display each decomposition over the test scene
- Discussion: can you replace the slant-range + terrain-correction chain with a GSLC-based one?

Complexity: advanced

##### ***About the test data:***

ESA BIOMASS Level-1 SCS (single-look complex, slant-range) products are **not** bundled in this repository — they are several GB each. Download a quad-polarimetric BIOMASS L1 SLC product from the [Copernicus Browser](https://dataspace.copernicus.eu/browser/) (registration required).

Look for filenames of the form `BIO_S2_SCS__1S_*`. Pick a forested scene (tropical, boreal or temperate) for the most informative decomposition outputs.

Place the unzipped product directory under the notebook's `data/` folder and update the path variable in the *Configure input paths* cell.

##### ***Some information on the Python environment:***

In [ ]:
import os
import sys
print("Python version: " + sys.version)

import sysconfig
print("Location of esa_snappy package: " + sysconfig.get_paths()['purelib'] + os.sep + "esa_snappy")

##### ***Import Python packages...***

In [ ]:
import esa_snappy
from esa_snappy import ProductIO

import snapista
from snapista import Graph
from snapista import Operator

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

##### ***Convenience plot functions:***

In [ ]:
def _read_band(product, name):
    band = product.getBand(name)
    if band is None:
        raise KeyError(f"Band {name!r} not found. Available: {[b.getName() for b in product.getBands()]}")
    w, h = band.getRasterWidth(), band.getRasterHeight()
    data = np.zeros(w * h, np.float32)
    band.readPixels(0, 0, w, h, data)
    data.shape = h, w
    return data

def plot_triplet(product, names, titles=None, cmap='viridis'):
    """Plot three bands side by side (e.g. entropy / anisotropy / alpha)."""
    if titles is None:
        titles = names
    fig, axs = plt.subplots(1, 3, figsize=(16, 5))
    for ax, name, title in zip(axs, names, titles):
        data = _read_band(product, name)
        im = ax.imshow(data, cmap=cmap)
        ax.set_title(title)
        fig.colorbar(im, ax=ax)
    plt.show()

def plot_pauli_rgb(product, r_band, g_band, b_band, percentile=98):
    """Plot a Pauli (or any 3-band) decomposition as RGB, percentile-stretched."""
    r = _read_band(product, r_band)
    g = _read_band(product, g_band)
    b = _read_band(product, b_band)
    rgb = np.stack([r, g, b], axis=-1)
    p = np.percentile(rgb, percentile)
    if p > 0:
        rgb = np.clip(rgb / p, 0, 1)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(rgb)
    ax.set_title('Pauli RGB: |HH−VV| (red), 2|HV| (green), |HH+VV| (blue)')
    plt.show()

def find_band(product, *patterns):
    """Return the first band whose name contains any of the given case-insensitive patterns."""
    names = [b.getName() for b in product.getBands()]
    for pat in patterns:
        for n in names:
            if pat.lower() in n.lower():
                return n
    raise KeyError(f"No band matching {patterns!r} found. Available: {names}")

---

### ***Background: ESA BIOMASS***

The **BIOMASS** mission (launched 2025) is the first spaceborne **P-band** synthetic aperture radar. P-band's long wavelength (≈70 cm, 435 MHz centre frequency) penetrates forest canopies far better than C-band or L-band, so the dominant scattering mechanism in forested areas is volume scattering from trunks and large branches — the components most directly related to above-ground biomass.

BIOMASS L1 SCS (Single-look Complex Slant range) products are **fully polarimetric** — they carry all four scattering matrix elements: HH, HV, VH, VV. This is the prerequisite for the polarimetric decompositions used below.

The SNAP Microwave Toolbox includes a dedicated `BiomassCalibrator` that applies the per-scene `calibration_factor` from the product metadata and produces calibrated complex bands suitable for downstream polarimetric processing. It is selected automatically by `Calibration` when the source product's `MISSION` attribute is `BIOMASS`.

---

### ***Background: the polarimetric processing chain***

The standard end-to-end SNAP polarimetric chain has five logical stages:

1. **Calibration** — convert the digital-number values (`i_*`, `q_*`) to calibrated sigma-nought-equivalent complex amplitudes, **keeping the complex (I+jQ) form** so downstream phase relationships survive. `Calibration` with `outputImageInComplex=true` does exactly this.

2. **Polarimetric matrix generation** (`Polarimetric-Matrices`) — combine the four calibrated complex channels into either the coherency matrix `T3` or the covariance matrix `C3`. These 3×3 Hermitian matrices fully characterise the polarimetric scattering at each pixel.

3. **Polarimetric speckle filtering** (`Polarimetric-Speckle-Filter`) — reduce speckle while preserving the matrix structure (i.e. without breaking the Hermitian / positive-semi-definite properties of T3/C3). **Refined Lee** is the workhorse; **IDAN**, **Lee Sigma**, **Box Car** and **Non-Local** are alternatives.

4. **Polarimetric decomposition** (`Polarimetric-Decomposition`) — factor T3/C3 into physically interpretable components. We demo three:
   - **H-A-Alpha** — Cloude-Pottier eigen-decomposition; outputs Entropy (H), Anisotropy (A) and mean Alpha angle. Generic scene characterisation.
   - **Pauli** — three orthogonal scattering mechanisms; displayed as RGB (`|HH−VV|`, `2|HV|`, `|HH+VV|`).
   - **Freeman-Durden** — physical 3-component model: surface, double-bounce, and volume scattering. The volume contribution is the proxy for forest biomass.

5. **Terrain correction** (`Terrain-Correction`) — geocode each derived band to a map projection using a DEM (Copernicus 30 m by default).

Order matters: decomposition must happen on T3/C3 **before** terrain correction, because terrain correction is a real-valued resampling that doesn't preserve the matrix structure.

---

### ***Configure input paths***

In [ ]:
data_dir = os.path.join(os.getcwd(), 'data')
graphs_dir = os.path.join(os.getcwd(), 'graphs')
results_dir = os.path.join(os.getcwd(), 'results')
os.makedirs(graphs_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

# --- cached data fetch (public S3 / HTTP; re-runs reuse the local copy) ---
import urllib.request as _urlreq, zipfile as _zip, glob as _glob

def fetch_cached(url, dest_dir):
    """Download `url` into dest_dir unless already present, unzip a .zip, and return the path to
    open (manifest.safe for a .SAFE product, else the downloaded file). Cached for re-runs:
    if the product is already in dest_dir it is NOT downloaded again."""
    os.makedirs(dest_dir, exist_ok=True)
    fname = url.split('/')[-1]
    stem = fname[:-4] if fname.lower().endswith('.zip') else fname
    hits = _glob.glob(os.path.join(dest_dir, stem + '*', 'manifest.safe'))
    if hits:
        print('cached:', os.path.basename(os.path.dirname(hits[0]))); return hits[0]
    local = os.path.join(dest_dir, fname)
    if not os.path.exists(local):
        print('downloading', fname, '...')
        _urlreq.urlretrieve(url, local)
        print('  saved %.0f MB' % (os.path.getsize(local) / 1e6))
    if fname.lower().endswith('.zip'):
        with _zip.ZipFile(local) as z:
            z.extractall(dest_dir)
        hits = _glob.glob(os.path.join(dest_dir, stem + '*', 'manifest.safe'))
        return hits[0] if hits else local
    return local

# BIOMASS L1 SCS product — adjust to your downloaded product
biomass_slc = os.path.join(data_dir, 'BIO_S2_SCS__1S_<acquisition>', 'manifest.safe')
# To auto-download inputs from a public bucket, use fetch_cached, e.g.:
#   product = fetch_cached('https://<bucket>/<product>.zip', data_dir)


---
## ***Part 1 — End-to-end chain with H-A-Alpha decomposition***
---

The graph below is the entire polarimetric chain in one piece. Snapista emits a single `gpt` graph that runs without intermediate read/write, so each tile flows through every operator before disk I/O.

Notable parameter choices:
- `Calibration`: `outputImageInComplex=true`, `outputSigmaBand=false` — keep the calibrated complex bands and skip the (redundant for polarimetry) sigma-nought intensity outputs.
- `Polarimetric-Matrices`: `matrix='T3'` — coherency matrix is the standard input for both eigen-based and model-based decompositions.
- `Polarimetric-Speckle-Filter`: **Refined Lee** with a 5×5 window — preserves edges and matrix structure.
- `Polarimetric-Decomposition`: **H-A-Alpha Quad Pol Decomposition** — outputs Entropy, Anisotropy and mean Alpha.
- `Terrain-Correction`: 25 m pixel spacing, WGS84(DD), Copernicus 30 m DEM.

In [ ]:
g_haa = Graph()

g_haa.add_node(operator=Operator('Read', file=biomass_slc), node_id='Read')

g_haa.add_node(operator=Operator('Calibration',
                                 outputImageInComplex='true',
                                 outputSigmaBand='false',
                                 selectedPolarisations='HH,HV,VH,VV'),
               node_id='Calib', source='Read')

g_haa.add_node(operator=Operator('Polarimetric-Matrices',
                                 matrix='T3'),
               node_id='Matrix', source='Calib')

g_haa.add_node(operator=Operator('Polarimetric-Speckle-Filter',
                                 filter='Refined Lee Filter',
                                 filterSize='5'),
               node_id='Speckle', source='Matrix')

g_haa.add_node(operator=Operator('Polarimetric-Decomposition',
                                 decomposition='H-A-Alpha Quad Pol Decomposition',
                                 outputHAAlpha='true'),
               node_id='Decomp', source='Speckle')

g_haa.add_node(operator=Operator('Terrain-Correction',
                                 demName='Copernicus 30m Global DEM',
                                 mapProjection='WGS84(DD)',
                                 pixelSpacingInMeter='25',
                                 imgResamplingMethod='BILINEAR_INTERPOLATION'),
               node_id='TC', source='Decomp')

haa_out = os.path.join(results_dir, 'snap_nb_biomass_polarimetric_haa.dim')
g_haa.add_node(operator=Operator('Write', file=haa_out, formatName='BEAM-DIMAP'),
               node_id='Write', source='TC')

g_haa.view()
g_haa.save_graph(os.path.join(graphs_dir, 'snap_nb_biomass_polarimetric_haa.xml'))
g_haa.run()

##### ***Display Entropy, Anisotropy, mean Alpha:***

In [ ]:
p_haa = ProductIO.readProduct(haa_out)
print('Bands:', [b.getName() for b in p_haa.getBands()])

# Band names follow SNAP's H-A-Alpha decomposition output naming.
# Adjust below if your SNAP version names them differently.
entropy = find_band(p_haa, 'Entropy')
anisotropy = find_band(p_haa, 'Anisotropy')
alpha = find_band(p_haa, 'Alpha')
plot_triplet(p_haa,
             names=[entropy, anisotropy, alpha],
             titles=['Entropy (H)', 'Anisotropy (A)', 'mean Alpha (°)'],
             cmap='viridis')
p_haa.dispose()

---
## ***Part 2 — Pauli decomposition (RGB)***
---

The Pauli decomposition expresses the complex scattering matrix as a sum of three orthogonal mechanisms:

- `|HH − VV|` — even-bounce / **double-bounce** scattering (urban, tree trunks bouncing off the ground)
- `2|HV|` — **volume** scattering (forest canopy)
- `|HH + VV|` — odd-bounce / **surface** scattering (smooth surfaces, open ground)

Mapping these three components to R, G, B respectively gives the iconic *Pauli RGB* — a hugely informative quick-look for any polarimetric scene.

We keep the same full chain so the output is directly comparable with the H-A-Alpha result above.

In [ ]:
g_pauli = Graph()
g_pauli.add_node(operator=Operator('Read', file=biomass_slc), node_id='Read')
g_pauli.add_node(operator=Operator('Calibration',
                                   outputImageInComplex='true',
                                   outputSigmaBand='false',
                                   selectedPolarisations='HH,HV,VH,VV'),
                 node_id='Calib', source='Read')
g_pauli.add_node(operator=Operator('Polarimetric-Matrices', matrix='T3'),
                 node_id='Matrix', source='Calib')
g_pauli.add_node(operator=Operator('Polarimetric-Speckle-Filter',
                                   filter='Refined Lee Filter',
                                   filterSize='5'),
                 node_id='Speckle', source='Matrix')
g_pauli.add_node(operator=Operator('Polarimetric-Decomposition',
                                   decomposition='Pauli Decomposition'),
                 node_id='Decomp', source='Speckle')
g_pauli.add_node(operator=Operator('Terrain-Correction',
                                   demName='Copernicus 30m Global DEM',
                                   mapProjection='WGS84(DD)',
                                   pixelSpacingInMeter='25',
                                   imgResamplingMethod='BILINEAR_INTERPOLATION'),
                 node_id='TC', source='Decomp')

pauli_out = os.path.join(results_dir, 'snap_nb_biomass_polarimetric_pauli.dim')
g_pauli.add_node(operator=Operator('Write', file=pauli_out, formatName='BEAM-DIMAP'),
                 node_id='Write', source='TC')

g_pauli.save_graph(os.path.join(graphs_dir, 'snap_nb_biomass_polarimetric_pauli.xml'))
g_pauli.run()

In [ ]:
p_pauli = ProductIO.readProduct(pauli_out)
print('Bands:', [b.getName() for b in p_pauli.getBands()])

# Pauli outputs are named Pauli_r / Pauli_g / Pauli_b in current SNAP versions.
r = find_band(p_pauli, 'Pauli_r', 'Pauli_R')
g = find_band(p_pauli, 'Pauli_g', 'Pauli_G')
b = find_band(p_pauli, 'Pauli_b', 'Pauli_B')
plot_pauli_rgb(p_pauli, r_band=r, g_band=g, b_band=b)
p_pauli.dispose()

---
## ***Part 3 — Freeman-Durden 3-component decomposition***
---

Freeman-Durden is a **model-based** decomposition: it assumes the scene is a mix of three scattering processes and solves for the power contribution of each.

- **Surface (Bragg) scattering** — bare soil, water
- **Double-bounce scattering** — trunk–ground interaction in forest, building–ground in urban
- **Volume scattering** — random orientation of cylinders, dominated by forest canopy

In a forested BIOMASS scene the **volume** component should dominate over canopy, with **double-bounce** flaring up near trunks and **surface** scattering visible on logging tracks and clearings.

In [ ]:
g_fd = Graph()
g_fd.add_node(operator=Operator('Read', file=biomass_slc), node_id='Read')
g_fd.add_node(operator=Operator('Calibration',
                                outputImageInComplex='true',
                                outputSigmaBand='false',
                                selectedPolarisations='HH,HV,VH,VV'),
              node_id='Calib', source='Read')
g_fd.add_node(operator=Operator('Polarimetric-Matrices', matrix='T3'),
              node_id='Matrix', source='Calib')
g_fd.add_node(operator=Operator('Polarimetric-Speckle-Filter',
                                filter='Refined Lee Filter',
                                filterSize='5'),
              node_id='Speckle', source='Matrix')
g_fd.add_node(operator=Operator('Polarimetric-Decomposition',
                                decomposition='Freeman-Durden Decomposition'),
              node_id='Decomp', source='Speckle')
g_fd.add_node(operator=Operator('Terrain-Correction',
                                demName='Copernicus 30m Global DEM',
                                mapProjection='WGS84(DD)',
                                pixelSpacingInMeter='25',
                                imgResamplingMethod='BILINEAR_INTERPOLATION'),
              node_id='TC', source='Decomp')

fd_out = os.path.join(results_dir, 'snap_nb_biomass_polarimetric_fd.dim')
g_fd.add_node(operator=Operator('Write', file=fd_out, formatName='BEAM-DIMAP'),
              node_id='Write', source='TC')

g_fd.save_graph(os.path.join(graphs_dir, 'snap_nb_biomass_polarimetric_fd.xml'))
g_fd.run()

In [ ]:
p_fd = ProductIO.readProduct(fd_out)
print('Bands:', [b.getName() for b in p_fd.getBands()])

# Freeman-Durden outputs are typically named Freeman_<component>_<r|g|b>.
surf = find_band(p_fd, 'Freeman_Surf', 'Surface')
dbl = find_band(p_fd, 'Freeman_Dbl', 'Double')
vol = find_band(p_fd, 'Freeman_Vol', 'Volume')
plot_triplet(p_fd,
             names=[surf, dbl, vol],
             titles=['Surface', 'Double-bounce', 'Volume'],
             cmap='magma')

# Show the same three components as an RGB composite
plot_pauli_rgb(p_fd, r_band=dbl, g_band=vol, b_band=surf)
p_fd.dispose()

---

### ***Side-note: can you do GSLC + polarimetric processing instead?***

The chain above is the classical SNAP polarimetric flow: polarimetric processing in **slant-range** geometry, with `Terrain-Correction` as the final step.

A natural question, given that the [GSLC notebook](snap-nb-sar-gslc-insar.ipynb) showed `GSLC-Terrain-Correction` can map-project a complex SLC while preserving phase, is:

> **Can we replace `slant-range polarimetric + final Terrain-Correction` with `GSLC-Terrain-Correction + map-projected polarimetric`?**

**In principle, yes** — and the appeal is the same as for InSAR: skip the final terrain-correction step entirely, work directly in a map projection, integrate cleanly with downstream GIS tools.

**In practice, today, it's not the recommended path.** The reasons:

1. **Operator validation.** `Polarimetric-Matrices`, `Polarimetric-Speckle-Filter` and `Polarimetric-Decomposition` were designed and validated against slant-range geometry. They will run on map-projected complex input, but the statistical decompositions implicitly assume locally stationary speckle, which depends on consistent pixel spacing — and that's only true *after* a careful Terrain-Correction. With GSLC-projected data the local statistics vary with terrain slope.

2. **Speckle-filter window geometry.** A 5×5 Refined Lee window in slant range covers a fixed `slant-range × azimuth` patch; the same window in map projection covers a varying ground area depending on incidence angle.

3. **Cross-polarisation phase.** `GSLC-Terrain-Correction` does preserve the phase relationships **between** polarisation channels when all four are processed together (which it does by default for a quad-pol product). So the data shape after `GSLC-Terrain-Correction` is, on paper, suitable for `Polarimetric-Matrices`.

**Practical recipe if you want to try it:**

```text
Read → Apply-Orbit-File → Calibration(outputImageInComplex=true)
     → GSLC-Terrain-Correction(outputFlattened=false)   ← keep the carrier phase
     → Polarimetric-Matrices(T3)
     → Polarimetric-Speckle-Filter(Refined Lee)
     → Polarimetric-Decomposition(...)
     → Write
```

Note **`outputFlattened=false`** here (which is also the default): polarimetric processing wants the full carrier-phase form. `outputFlattened=true` strips the geometric carrier and is not useful for polarimetry (nor for InSAR).

For now the safer path for production polarimetric work is the classical chain shown in Parts 1–3 above.

---

### ***Summary***

What have we learnt in this notebook?

- **BIOMASS** is a fully polarimetric P-band SAR mission whose volume-scattering signature is well-suited for above-ground biomass retrieval.
- The standard SNAP polarimetric chain is `Calibration → Polarimetric-Matrices → Polarimetric-Speckle-Filter → Polarimetric-Decomposition → Terrain-Correction`, with calibration kept in complex form (`outputImageInComplex=true`).
- `Polarimetric-Matrices` with `matrix='T3'` produces the 3×3 coherency matrix used by both eigen-based (H-A-Alpha, Cloude) and model-based (Freeman-Durden, Yamaguchi, vanZyl) decompositions.
- We ran three decompositions on the same BIOMASS scene:
  - **H-A-Alpha** for generic scene characterisation (Entropy, Anisotropy, Alpha)
  - **Pauli** for an RGB quick-look
  - **Freeman-Durden** for surface / double-bounce / volume separation
- Terrain correction is the **last** step: it's a real-valued resampling that doesn't preserve matrix structure, so it must come after decomposition.
- `GSLC-Terrain-Correction` is in principle compatible with polarimetric processing (with `outputFlattened=false`), but the standard polarimetric operators are validated against slant-range data — the classical chain is the safer choice today.